# Orchestrator-Worker Pattern

**Pattern**: Dynamic task assignment where an orchestrator decides which workers to spawn at runtime.

**Example**: Student Performance Evaluator

**Workflow**:
```
                         ┌→ [Process Student 1] ─┐
START → [Fan-Out] ──────┼→ [Process Student 2] ─┼→ [Merge Results] → END
                         └→ [Process Student N] ─┘
```

**Key Concept**: Uses `Send` API to dynamically spawn worker nodes based on input data.
- Number of workers is determined at runtime (not hardcoded)
- Each worker processes one student independently
- Results are merged using a reducer

## Setup

In [ ]:
from typing import TypedDict, List, Annotated
import operator
import json
import re

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from pydantic import BaseModel
from langchain_aws import ChatBedrockConverse

## Initialize LLM

In [ ]:
llm = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    temperature=0,
)

## Define Data Models

In [ ]:
# Input model
class Student(TypedDict):
    name: str
    project_work: str
    certification: str
    extra_work: str

# Output models (Pydantic for LLM structured output)
class FeedbackModel(BaseModel):
    overall_feedback: str
    core_strength: str
    area_of_improvement: str

class EvaluationOutput(BaseModel):
    review: str
    feedback: FeedbackModel

# Result stored in state
class StudentResult(TypedDict):
    name: str
    review: str
    feedback: dict

## Define State

Using `Annotated` with `operator.add` reducer so parallel workers can all append results.

In [ ]:
class ReviewState(TypedDict):
    students: List[Student]
    results: Annotated[List[StudentResult], operator.add]  # Reducer merges parallel outputs

## Define Worker Node

Each worker processes one student and returns evaluation.

In [ ]:
def build_prompt(student: Student) -> str:
    """Build evaluation prompt for a student."""
    return f"""You are an evaluator assessing student performance.

Return ONLY valid JSON matching this schema (no extra text):
{json.dumps(EvaluationOutput.model_json_schema(), indent=2)}

Student Information:
- Name: {student['name']}
- Project Work: {student['project_work']}
- Certification: {student['certification']}
- Extra Work: {student.get('extra_work', 'None')}

Evaluate the student and provide:
1. A brief review (2-3 sentences)
2. Feedback with overall assessment, core strength, and area of improvement"""

In [ ]:
def process_student(state: dict) -> dict:
    """Worker node: Process a single student."""
    student = state["student"]
    
    # Call LLM
    response = llm.invoke(build_prompt(student))
    content = response.content
    
    # Parse response
    try:
        parsed = EvaluationOutput.model_validate_json(content)
    except Exception:
        # Fallback: extract JSON from response
        json_match = re.search(r"\{.*\}", content, re.DOTALL)
        if json_match:
            parsed = EvaluationOutput.model_validate_json(json_match.group())
        else:
            # Default response if parsing fails
            parsed = EvaluationOutput(
                review="Unable to generate review",
                feedback=FeedbackModel(
                    overall_feedback="Evaluation pending",
                    core_strength="Not determined",
                    area_of_improvement="Not determined"
                )
            )
    
    return {
        "results": [{
            "name": student["name"],
            "review": parsed.review,
            "feedback": parsed.feedback.model_dump()
        }]
    }

## Define Orchestrator (Fan-Out Function)

Uses `Send` API to dynamically spawn workers based on number of students.

In [ ]:
def fan_out_students(state: ReviewState) -> List[Send]:
    """
    Orchestrator: Dynamically spawn a worker for each student.
    
    Returns a list of Send objects, each directing to 'process_student' node
    with a specific student's data.
    """
    return [
        Send("process_student", {"student": student})
        for student in state["students"]
    ]

## Build the Graph

In [ ]:
# Create graph
builder = StateGraph(ReviewState)

# Add worker node
builder.add_node("process_student", process_student)

# Add conditional edges from START using fan-out function
# This dynamically creates edges to process_student for each student
builder.add_conditional_edges(START, fan_out_students)

# All workers lead to END (results merged via reducer)
builder.add_edge("process_student", END)

# Compile
graph = builder.compile()

## Visualize the Graph

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Helper: Display Results

In [ ]:
def display_results(result: dict) -> None:
    """Pretty print evaluation results."""
    print("=" * 70)
    print("              STUDENT PERFORMANCE EVALUATION")
    print("=" * 70)
    print(f"\nTotal Students Evaluated: {len(result['results'])}")
    
    for i, student_result in enumerate(result['results'], 1):
        print("\n" + "-" * 70)
        print(f"STUDENT {i}: {student_result['name']}")
        print("-" * 70)
        
        print(f"\n[REVIEW]")
        print(f"  {student_result['review']}")
        
        feedback = student_result['feedback']
        print(f"\n[OVERALL FEEDBACK]")
        print(f"  {feedback['overall_feedback']}")
        
        print(f"\n[CORE STRENGTH]")
        print(f"  {feedback['core_strength']}")
        
        print(f"\n[AREA OF IMPROVEMENT]")
        print(f"  {feedback['area_of_improvement']}")
    
    print("\n" + "=" * 70)

## Test Case 1: Single Student

In [ ]:
result = graph.invoke({
    "students": [
        {
            "name": "Rahul",
            "project_work": "AI chatbot using LangChain",
            "certification": "AWS Machine Learning Specialty",
            "extra_work": ""
        }
    ],
    "results": []
})

display_results(result)

## Test Case 2: Multiple Students (Parallel Processing)

In [ ]:
result = graph.invoke({
    "students": [
        {
            "name": "Priya",
            "project_work": "E-commerce recommendation system",
            "certification": "Google Cloud Professional ML Engineer",
            "extra_work": "Open source contributions to scikit-learn"
        },
        {
            "name": "Amit",
            "project_work": "Sentiment analysis dashboard",
            "certification": "None",
            "extra_work": "Kaggle competitions - Bronze medal"
        },
        {
            "name": "Sneha",
            "project_work": "Computer vision for quality control",
            "certification": "Azure AI Engineer Associate",
            "extra_work": "Technical blog writing, mentoring juniors"
        }
    ],
    "results": []
})

display_results(result)

## Test Case 3: Varied Student Profiles

In [ ]:
result = graph.invoke({
    "students": [
        {
            "name": "Vikram",
            "project_work": "Basic CRUD application",
            "certification": "None",
            "extra_work": "None"
        },
        {
            "name": "Ananya",
            "project_work": "Full-stack AI platform with RAG, agents, and multi-modal support",
            "certification": "AWS Solutions Architect, Google Cloud ML, Azure AI",
            "extra_work": "Published research paper, 5000+ GitHub stars, conference speaker"
        }
    ],
    "results": []
})

display_results(result)

## Stream Execution

In [ ]:
print("=" * 70)
print("STREAMING EXECUTION")
print("=" * 70)

for step in graph.stream({
    "students": [
        {"name": "Student A", "project_work": "ML Pipeline", "certification": "AWS", "extra_work": ""},
        {"name": "Student B", "project_work": "Web App", "certification": "None", "extra_work": "Blog"}
    ],
    "results": []
}):
    for node_name, output in step.items():
        if "results" in output and output["results"]:
            student_name = output["results"][0]["name"]
            print(f"\n[DONE] Worker completed: {student_name}")
            print(f"   Review: {output['results'][0]['review'][:80]}...")

## Key Takeaways

1. **Orchestrator-Worker** uses `Send` API for dynamic task assignment
2. **Number of workers is runtime-determined** - depends on input data size
3. **Workers run in parallel** - each processes independently
4. **Reducer merges results** - `Annotated[List, operator.add]` combines all outputs
5. **Pydantic models** ensure structured LLM output

### Send API Pattern:
```python
from langgraph.types import Send

def fan_out(state):
    return [
        Send("worker_node", {"item": item})
        for item in state["items"]
    ]

graph.add_conditional_edges(START, fan_out)
```

### Difference from Parallelisation Pattern:
| Aspect | Parallelisation | Orchestrator-Worker |
|--------|-----------------|--------------------|
| Workers | Fixed at design time | Dynamic at runtime |
| Edges | Hardcoded parallel edges | `Send` creates edges dynamically |
| Use case | Known fixed subtasks | Variable number of items to process |